# Standings to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/standings/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/standings/standings"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/standings"
TABLE      = "nfl.standings"

In [ ]:
# Use a minimal schema for the standings array (dynamic fields flatten at read time)
file_schema = StructType([
    StructField("teamId",      StringType()),
    StructField("teamName",    StringType()),
    StructField("season",      StringType()),
    StructField("conference",  StringType()),
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    teamId        STRING    COMMENT 'Team ID',
    teamName      STRING,
    season        STRING    COMMENT 'NFL season year',
    conference    STRING,
    stats         MAP<STRING, STRING> COMMENT 'Dynamic ESPN stats (wins, losses, etc)',
    snapshot_at   TIMESTAMP COMMENT 'When the snapshot file was written',
    source_file   STRING,
    ingested_at   TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
# Read JSON as untyped to capture dynamic stats fields
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

# Extract known fields and capture all other fields as a stats map
standings = (raw
    .select(
        F.col("teamId"),
        F.col("teamName"),
        F.col("season"),
        F.col("conference"),
        # Capture all fields except the known ones as a map
        F.map_from_arrays(
            F.filter(F.map_keys(F.from_json(F.to_json(F.struct("*")), "MAP<STRING,STRING>")), 
                     lambda x: ~F.isin(x, ["teamId", "teamName", "season", "conference", "source_file", "snapshot_at"])),
            F.filter(F.map_values(F.from_json(F.to_json(F.struct("*")), "MAP<STRING,STRING>")), 
                     lambda x: x.isNotNull())
        ).alias("stats"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    ))

In [ ]:
# Simpler approach: use schema_of_json and cast all unknown fields as strings
# This approach works better with Databricks

standings_fixed = (raw
    .select(
        F.col("teamId"),
        F.col("teamName"),
        F.col("season"),
        F.col("conference"),
        # Convert entire row to JSON then extract all fields as a map of strings
        F.from_json(F.to_json(F.struct("*")), "MAP<STRING, STRING>").alias("all_fields"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    )
    .select(
        F.col("teamId"),
        F.col("teamName"),
        F.col("season"),
        F.col("conference"),
        # Drop known fields from the map, keep the rest as stats
        F.map_filter(
            F.col("all_fields"),
            lambda k, v: ~F.isin(k, ["teamId", "teamName", "season", "conference", "source_file", "snapshot_at", "ingested_at"])
        ).alias("stats"),
        "snapshot_at",
        "source_file",
        "ingested_at",
    ))

In [ ]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # Step 1: Dedupe within batch (keep newest snapshot per team/season)
    w = Window.partitionBy("teamId", "season").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))
    
    # Step 2: Auto-add missing columns (schema evolution)
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # Step 3: Merge into table
    (target_table.alias("t")
        .merge(latest.alias("s"), "t.teamId = s.teamId AND t.season = s.season")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(standings_fixed.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM nfl.standings;

SELECT teamId, teamName, season, conference, stats FROM nfl.standings ORDER BY season DESC, teamName LIMIT 10;